# BasketWise walkthrough

This notebook runs the BasketWise pipeline step by step with the `market_basket` package: load and validate transactions, explore products, mine association rules, recommend products, find bundle opportunities and evaluate the recommendations against a best-seller baseline.

* The data is **synthetic**: generated by `scripts/generate_data.py` with a fixed seed. It does not describe real customers or sales.
* Frequent itemsets (Apriori and FP-Growth) and association rules are computed by the **mlxtend** library. Everything around them (validation, ranking, recommendations, evaluation, insights) is BasketWise code.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # import the package from the project root

import pandas as pd

from market_basket import plots
from market_basket.config import MiningConfig
from market_basket.data import item_support, load_demo_data, summarize
from market_basket.evaluation import evaluate
from market_basket.insights import bundle_opportunities, cross_sell_opportunities, product_groups, product_popularity
from market_basket.mining import explain_rule, mine_rules, readable
from market_basket.recommend import Recommender, compare_rankings, recommendations_table

pd.set_option("display.precision", 4)

## 1. Load and validate the data

The loader detects the file layout, trims spaces, merges capitalisation variants and duplicate rows, and records every removed row with a reason. The demo data is clean, so nothing is removed.

In [2]:
data = load_demo_data(Path.cwd().parent / "Data" / "synthetic_retail_transactions.csv")
pd.Series(summarize(data), name="value").to_frame()

,value
transactions,8000
unique products,47
categories,8
product lines,29763
average basket size,3.72
median basket size,4.0
largest basket,13
single-product share,0.1227
first date,2025-01-01
last date,2025-06-30


In [3]:
data.report.summary_table()

,Check,Result
0,File layout,long: one row per product
1,Total product lines (rows) received,"29,763"
2,Valid transactions,"8,000"
3,Valid product lines,"29,763"
4,Unique products,47
5,Rejected: missing transaction ID,0
6,Removed: blank product names,0
7,Removed: duplicate product in a transaction,0
8,Removed: transactions with no products,0
9,Fixed: extra spaces,0


In [4]:
data.items.head()

,transaction_id,product_name,product_id,category,quantity,unit_price,transaction_date,customer_id
0,SYN-T000001,Paneer,P006,Dairy & Eggs,1,90.0,2025-01-01,SYN-C00732
1,SYN-T000001,Onions,P024,Fruits & Vegetables,1,35.0,2025-01-01,SYN-C00732
2,SYN-T000001,Tomatoes,P025,Fruits & Vegetables,1,30.0,2025-01-01,SYN-C00732
3,SYN-T000001,Apples,P023,Fruits & Vegetables,1,160.0,2025-01-01,SYN-C00732
4,SYN-T000002,Toothpaste,P046,Personal Care,1,95.0,2025-01-01,SYN-C00759


## 2. Products and baskets

In [5]:
popularity = product_popularity(data)
plots.product_bar(popularity, top_n=15)

In [6]:
plots.basket_size_chart(data)

## 3. Choosing the thresholds

Minimum support decides how often a product combination must occur before it is trusted. The table shows how many itemsets and rules each level produces (confidence ≥ 20 %, lift ≥ 1.2, up to 3 products).

In [7]:
rows = []
for min_support in [0.05, 0.02, 0.01, 0.005]:
    result = mine_rules(data.transactions, MiningConfig(min_support=min_support))
    lengths = result.frequent_itemsets["length"]
    rows.append({"min support": min_support, "frequent itemsets": len(lengths),
                 "pairs": int((lengths == 2).sum()), "triples": int((lengths == 3).sum()),
                 "rules": len(result.rules)})
pd.DataFrame(rows)

,min support,frequent itemsets,pairs,triples,rules
0,0.050,40,9,0,18
1,0.020,113,56,11,113
2,0.010,265,176,42,262
3,0.005,623,400,176,571


## 4. Association rules with the default settings

In [8]:
result = mine_rules(data.transactions, MiningConfig())
rules = result.rules
readable(rules)[["antecedents", "consequents", "support", "confidence", "lift"]].head(10)

,antecedents,consequents,support,confidence,lift
0,"Cheese, Pasta",Pasta Sauce,0.0144,0.5928,14.1984
1,Pasta Sauce,"Cheese, Pasta",0.0144,0.3443,14.1984
2,"Shampoo, Soap",Conditioner,0.0105,0.5217,13.7300
3,Conditioner,"Shampoo, Soap",0.0105,0.2763,13.7300
4,"Conditioner, Soap",Shampoo,0.0105,0.9655,13.6953
5,"Cheese, Pasta Sauce",Pasta,0.0144,0.9914,13.2405
6,"Pasta, Tomatoes",Pasta Sauce,0.0116,0.5314,12.7288
7,Pasta Sauce,"Pasta, Tomatoes",0.0116,0.2784,12.7288
8,"Detergent, Dishwash Liquid",Fabric Softener,0.0107,0.5548,12.5034
9,Fabric Softener,"Detergent, Dishwash Liquid",0.0107,0.2423,12.5034


In [9]:
top = rules.iloc[0]
print(explain_rule(top["antecedents"], top["consequents"], top["support"], top["confidence"],
                   top["lift"], top["consequent support"]))

In the observed transactions, 59% of baskets with Cheese and Pasta also contained Pasta Sauce. That is 14.20 times the overall rate of Pasta Sauce (4%). Both appear together in 1.4% of all transactions.


### Do the rules recover the planted patterns?

Because the data is synthetic, we know which associations were planted (for example a *pasta night* mission that adds pasta sauce to baskets with pasta) and which products are unrelated (pasta and shampoo). This checks that the pipeline finds real structure and does not invent links.

In [10]:
def lift(antecedent, consequent):
    match = rules[(rules["antecedents"] == frozenset({antecedent})) & (rules["consequents"] == frozenset({consequent}))]
    return round(float(match.iloc[0]["lift"]), 2) if len(match) else "no rule (below the thresholds)"

pd.Series({"Pasta -> Pasta Sauce (planted)": lift("Pasta", "Pasta Sauce"),
           "Shampoo -> Conditioner (planted)": lift("Shampoo", "Conditioner"),
           "Pasta -> Shampoo (unrelated)": lift("Pasta", "Shampoo")}, name="lift").to_frame()

,lift
Pasta -> Pasta Sauce (planted),10.16
Shampoo -> Conditioner (planted),11.43
Pasta -> Shampoo (unrelated),no rule (below the thresholds)


In [11]:
plots.rules_scatter(rules)

## 5. Recommendations

A rule applies when all of its antecedent products are in the basket. Each product is suggested once, through its best rule (confidence first by default), with a reason. Best sellers fill any free slots.

In [12]:
best_sellers = item_support(data.transactions)
recommender = Recommender(rules, "confidence", best_sellers)
recommendations_table(recommender.recommend(["Milk", "Bread"], top_n=5))

,rank,recommended product,source,based on,support,confidence,lift,reason
0,1,Butter,association rule,"Bread, Milk",0.0409,0.4088,3.7500,Customers whose basket contained Bread and Mil...
1,2,Eggs,association rule,"Bread, Milk",0.0396,0.3962,2.8950,Customers whose basket contained Bread and Mil...
2,3,Jam,association rule,"Bread, Milk",0.0288,0.2875,5.0885,Customers whose basket contained Bread and Mil...
3,4,Tea,association rule,Milk,0.0665,0.2525,1.6805,Customers whose basket contained Milk also bou...
4,5,Bananas,best seller,,NaN,NaN,NaN,"Best seller, bought in 16% of all transactions..."


In [13]:
compare_rankings(rules, ["Milk", "Bread"], best_sellers, top_n=5)

,ranked by confidence,ranked by lift,ranked by support
rank,,,
1,Butter,Jam,Tea
2,Eggs,Butter,Butter
3,Jam,Eggs,Eggs
4,Tea,Tea,Jam
5,Bananas (best seller),Bananas (best seller),Bananas (best seller)


## 6. Bundles, cross-selling and merchandising groups

In [14]:
bundle_opportunities(result, best_sellers, data.categories, top_n=10)

,products,size,transactions together,support,all-confidence,lift vs independence,categories,potential use
0,Conditioner + Shampoo,2,245,0.0306,0.4344,11.4315,Personal Care,Shelve together within Personal Care
1,Biscuits + Tea,2,514,0.0643,0.4276,3.8395,Bakery + Beverages,Cross-category display or bundle offer
2,Pasta + Pasta Sauce,2,254,0.0318,0.4240,10.1566,Staples & Cooking,Shelve together within Staples & Cooking
3,Detergent + Fabric Softener,2,277,0.0346,0.3805,8.5745,Household,Shelve together within Household
4,Bread + Milk,2,800,0.1000,0.3797,1.7558,Bakery + Dairy & Eggs,Cross-category display or bundle offer
5,Potato Chips + Soft Drink,2,371,0.0464,0.3706,3.9853,Snacks + Beverages,Cross-category display or bundle offer
6,Onions + Rice,2,460,0.0575,0.3662,2.5021,Fruits & Vegetables + Staples & Cooking,Cross-category display or bundle offer
7,Lentils + Rice,2,439,0.0549,0.3495,4.1796,Staples & Cooking,Shelve together within Staples & Cooking
8,Rice + Tomatoes,2,427,0.0534,0.3400,2.3857,Staples & Cooking + Fruits & Vegetables,Cross-category display or bundle offer
9,Cheese + Pasta,2,194,0.0243,0.3239,6.5429,Dairy & Eggs + Staples & Cooking,Cross-category display or bundle offer


In [15]:
cross_sell_opportunities(rules, data.categories, top_n=10)

,when a basket has,suggest,support,confidence,lift,categories
0,Conditioner,Shampoo,0.0306,0.8059,11.4315,Personal Care
1,Pasta Sauce,Pasta,0.0318,0.7605,10.1566,Staples & Cooking
2,Toothbrush,Toothpaste,0.0171,0.5983,9.1511,Personal Care
3,Fabric Softener,Detergent,0.0346,0.7803,8.5745,Household
4,Dishwash Liquid,Fabric Softener,0.0112,0.3226,7.2694,Household
5,Croissant,Coffee,0.0177,0.6201,7.0665,Bakery + Beverages
6,Pasta Sauce,Cheese,0.0145,0.3473,7.0163,Staples & Cooking + Dairy & Eggs
7,Cheese,Pasta,0.0243,0.4899,6.5429,Dairy & Eggs + Staples & Cooking
8,Spinach,Paneer,0.0114,0.3569,6.2334,Fruits & Vegetables + Dairy & Eggs
9,Dishwash Liquid,Detergent,0.0194,0.5556,6.1050,Household


In [16]:
product_groups(rules, min_lift=2.0)

[['Cooking Oil',
  'Lentils',
  'Onions',
  'Paneer',
  'Potatoes',
  'Rice',
  'Spice Mix',
  'Spinach',
  'Tomatoes'],
 ['Biscuits', 'Coffee', 'Croissant', 'Sugar', 'Tea', 'Wheat Flour'],
 ['Apples', 'Bananas', 'Lemons', 'Orange Juice', 'Yogurt'],
 ['Bread', 'Butter', 'Eggs', 'Jam', 'Milk'],
 ['Chocolate', 'Popcorn', 'Potato Chips', 'Salted Peanuts', 'Soft Drink'],
 ['Detergent', 'Dishwash Liquid', 'Fabric Softener', 'Garbage Bags'],
 ['Cheese', 'Pasta', 'Pasta Sauce'],
 ['Conditioner', 'Shampoo', 'Soap'],
 ['Toothbrush', 'Toothpaste']]

## 7. Evaluation against the best-seller baseline

For five random 80/20 splits: rules and best sellers come from the training split only; in every test transaction with at least two products each product is hidden in turn, and a method scores a hit if the hidden product is in its top 3 (or top 5) suggestions.

In [17]:
evaluation = evaluate(data.transactions, MiningConfig(), n_splits=5, seed=42)
evaluation.summary

,method,cases,hit rate@3,hit rate@5,availability,coverage,hit rate@3 std,hit rate@5 std,splits beating baseline@3,splits beating baseline@5
0,Best-seller baseline,5760,0.1971,0.3006,1.0000,0.2596,0.0058,0.0054,<NA>,<NA>
1,Rules ranked by confidence,5760,0.4252,0.5024,0.9876,0.7660,0.0022,0.0012,5,5
2,Rules ranked by lift,5760,0.3841,0.4819,0.9876,0.7660,0.0049,0.0048,5,5
3,Rules by confidence + best-seller fallback,5760,0.4348,0.5308,1.0000,0.7660,0.0017,0.0018,5,5


In [18]:
plots.evaluation_chart(evaluation.summary)

In [19]:
summary = evaluation.summary.set_index("method")
baseline = summary.loc["Best-seller baseline", "hit rate@3"]
for method, row in summary.drop(index="Best-seller baseline").iterrows():
    print(f"{method}: hit rate@3 {row['hit rate@3']:.1%} vs {baseline:.1%} for best sellers "
          f"({(row['hit rate@3'] - baseline) * 100:+.1f} points)")

Rules ranked by confidence: hit rate@3 42.5% vs 19.7% for best sellers (+22.8 points)
Rules ranked by lift: hit rate@3 38.4% vs 19.7% for best sellers (+18.7 points)
Rules by confidence + best-seller fallback: hit rate@3 43.5% vs 19.7% for best sellers (+23.8 points)


## Limitations

* The associations were planted by the data generator, so rules are expected to beat the baseline here. Real retail data is noisier and the gap may be much smaller or reversed.
* Offline hit rate measures whether suggestions match what customers bought anyway, not extra sales. Business impact needs a controlled experiment such as an A/B test.
* Rules describe co-occurrence, not cause and effect; the train/test split is random rather than by time.